# Sort text into categories

**The job.** Short messages. Work out which of three teams each one belongs to.

No model downloads. Bag of words and a linear classifier, in numpy, so you can
read every line of the maths.

Feature building splits in two — the words themselves, and simple shape signals
like length and whether there is a question mark — and they meet at the
assemble step. Same diamond as the tabular notebook, different domain.

In [ ]:
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import json, pathlib
from dataclasses import replace

from browsergraph import execute, viz
from browsergraph.compile import compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import Edge, NodeCandidate, StageDefinition, WorkbenchDefinition

# A fresh folder each run. Left-over files from a previous run make the "what
# did this produce" list a lie, and that list is half the point here.
import shutil
WORK = pathlib.Path("work")
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

# These come from the library rather than being redefined in every notebook.
# They used to be thirty lines pasted into each one, which meant anyone copying
# a notebook to start a project got helpers that did not exist in browsergraph.
from browsergraph.quick import chain, fanin, fanout, link, node, problems, step
from browsergraph.quick import graph as _graph
from browsergraph.quick import subgraph  # noqa: F401  (used by later notebooks)

# The notebooks kept the older names, and `build` also prints what is wrong
# rather than raising — in a notebook the complaint is the lesson.
stage = step

def build(title, task, stages, nodes, edges=()):
    bench = _graph(title, task, stages, nodes, edges)
    print("problems:", problems(bench) or "none")
    return bench

print("ready")

In [ ]:
MESSAGES = [
    ("billing",  "my invoice is wrong again"),
    ("billing",  "charged twice this month"),
    ("billing",  "can I get a refund for the duplicate charge"),
    ("billing",  "the invoice total does not match my order"),
    ("billing",  "please cancel my subscription and refund"),
    ("billing",  "why was I charged after cancelling"),
    ("access",   "cannot log in to my account"),
    ("access",   "password reset email never arrives"),
    ("access",   "locked out after too many attempts"),
    ("access",   "two factor code is not accepted"),
    ("access",   "my login stopped working today"),
    ("access",   "reset link expired before I could use it"),
    ("bug",      "the export button does nothing"),
    ("bug",      "page crashes when I upload a file"),
    ("bug",      "the report shows blank rows"),
    ("bug",      "app freezes on the settings screen"),
    ("bug",      "clicking save throws an error"),
    ("bug",      "the chart renders upside down"),
]
HOLD_OUT = [
    ("billing", "I was charged twice for one invoice"),
    ("access",  "cannot reset my password"),
    ("bug",     "the upload page crashes every time"),
]
print(f"{len(MESSAGES)} training messages, {len(HOLD_OUT)} held back")
for label, message in MESSAGES[:3]:
    print(f"  {label:<9}{message}")

In [ ]:
nodes = [
    node("load.msgs",   "read",     [],                  [("out", "Corpus")]),
    node("words.bag",   "words",    [("in", "Corpus")],  [("out", "Matrix")]),
    node("shape.simple","shape",    [("in", "Corpus")],  [("out", "Matrix")]),
    node("join.side",   "assemble", [("words", "Matrix"), ("shape", "Matrix")], [("out", "Matrix")]),
    node("fit.softmax", "fit",      [("in", "Matrix")],  [("out", "Model")]),
    node("score.held",  "score",    [("in", "Model")],   [("out", "Score")]),
]

stages = [
    stage("load",     "Load messages",   [],                 [("out", "Corpus")], "read",  ["load.msgs"]),
    stage("words",    "Count words",     [("in", "Corpus")], [("out", "Matrix")], "words", ["words.bag"]),
    stage("shape",    "Shape signals",   [("in", "Corpus")], [("out", "Matrix")], "shape", ["shape.simple"]),
    stage("assemble", "Put together",    [("words", "Matrix"), ("shape", "Matrix")], [("out", "Matrix")], "assemble", ["join.side"]),
    stage("fit",      "Fit",             [("in", "Matrix")], [("out", "Model")],  "fit",   ["fit.softmax"]),
    stage("score",    "Score held-out",  [("in", "Model")],  [("out", "Score")],  "score", ["score.held"]),
]

edges = [Edge("load", "words"), Edge("load", "shape"),
         Edge("words", "assemble", to_port="words"),
         Edge("shape", "assemble", to_port="shape"),
         Edge("assemble", "fit"), Edge("fit", "score")]

bench = build("Sort text into categories",
              "Put each message with the team that should read it.", stages, nodes, edges)
print("layers:", bench.layers())

In [ ]:
viz.dag(bench)

In [ ]:
import numpy as np

LABELS = ["billing", "access", "bug"]

def load_msgs():
    return {"train": MESSAGES, "test": HOLD_OUT}

def words_bag(**kw):
    """One column per word that shows up at least twice."""
    corpus = kw["in"]["train"]
    counts = {}
    for _, message in corpus:
        for word in message.lower().split():
            counts[word] = counts.get(word, 0) + 1
    vocab = sorted(w for w, c in counts.items() if c >= 2)

    def vectorise(rows):
        M = np.zeros((len(rows), len(vocab)))
        for i, (_, message) in enumerate(rows):
            words = message.lower().split()
            for j, word in enumerate(vocab):
                M[i, j] = words.count(word)
        return M

    return {"train": vectorise(corpus), "test": vectorise(kw["in"]["test"]),
            "vocab": vocab}

def shape_simple(**kw):
    """Length and punctuation. Nothing to do with which words were used."""
    def shape(rows):
        return np.array([[len(m.split()), len(m), float("?" in m)] for _, m in rows])
    return {"train": shape(kw["in"]["train"]), "test": shape(kw["in"]["test"])}

def join_side(**kw):
    words, shape = kw["words"], kw["shape"]
    return {"train": np.column_stack([np.ones(len(words["train"])), words["train"], shape["train"]]),
            "test": np.column_stack([np.ones(len(words["test"])), words["test"], shape["test"]]),
            "vocab": words["vocab"],
            "y_train": np.array([LABELS.index(l) for l, _ in MESSAGES]),
            "y_test": np.array([LABELS.index(l) for l, _ in HOLD_OUT])}

def fit_softmax(**kw):
    """Multi-class logistic regression by gradient descent."""
    d = kw["in"]
    X, y = d["train"], d["y_train"]
    W = np.zeros((X.shape[1], len(LABELS)))
    onehot = np.eye(len(LABELS))[y]
    for _ in range(600):
        scores = X @ W
        scores -= scores.max(1, keepdims=True)
        probs = np.exp(scores); probs /= probs.sum(1, keepdims=True)
        W -= 0.35 * (X.T @ (probs - onehot)) / len(y)
    return {"W": W, "data": d}

def score_held(**kw):
    m = kw["in"]; d = m["data"]
    def predict(X):
        s = X @ m["W"]
        return s.argmax(1)
    train_pred, test_pred = predict(d["train"]), predict(d["test"])
    return {"train_accuracy": float((train_pred == d["y_train"]).mean()),
            "test_accuracy": float((test_pred == d["y_test"]).mean()),
            "predictions": [LABELS[i] for i in test_pred],
            "truth": [LABELS[i] for i in d["y_test"]],
            "vocab_size": len(d["vocab"])}

runtime = execute.Runtime({
    "load.msgs": load_msgs, "words.bag": words_bag, "shape.simple": shape_simple,
    "join.side": join_side, "fit.softmax": fit_softmax, "score.held": score_held})

plan = compile_route(bench, {s.id: s.candidates[0] for s in bench.leaf_stages})
run = execute.run(plan, runtime, workers=2)
print(run.text())

In [ ]:
got = run.output("score")
print(f"vocabulary: {got['vocab_size']} words")
print(f"training accuracy: {got['train_accuracy']:.0%}")
print(f"held-out accuracy: {got['test_accuracy']:.0%}\n")

for (truth, message), predicted in zip(HOLD_OUT, got["predictions"]):
    mark = "ok " if truth == predicted else "NO "
    print(f"  {mark} {predicted:<9} (really {truth:<9}) {message}")

Training accuracy of 100% on eighteen messages means very little. The held-out
three are the only honest number here, and three is far too few to trust.

Saying that is the point. A notebook that printed 100% and stopped would be
reporting the sample it fitted to.